In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool    
from langchain.agents import create_agent
load_dotenv()

c:\Users\kashy\OneDrive\Desktop\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
import json
with open("career_data.json") as f:
    career_data=json.load(f)


In [6]:
@tool
def search_career(role:str)->str:
    """Search career roles."""

    matches=[]

    for career in career_data:

        if role.lower() in career["role"].lower():
            matches.append(career)

    if not matches:
        return "Career not found."

    return str(matches)

In [7]:
@tool
def salary_info(role:str)->str:
    """Return salary information."""

    for career in career_data:

        if role.lower() in career["role"].lower():
            return career["salary"]

    return "Salary information not available."

In [8]:
@tool
def roadmap_generator(role:str)->str:
    """Generate learning roadmap."""

    for career in career_data:

        if role.lower() in career["role"].lower():

            return "\n".join(career["roadmap"])

    return "Roadmap not found."

In [9]:
@tool
def recommend_courses(role:str)->str:
    """Recommend courses."""

    for career in career_data:

        if role.lower() in career["role"].lower():

            return "\n".join(career["courses"])

    return "Courses not found."

In [10]:
@tool
def skill_gap(role:str,current_skills:str)->str:
    """Compare user skills with required skills."""

    user_skills=[s.strip().lower() for s in current_skills.split(",")]

    for career in career_data:

        if role.lower() in career["role"].lower():

            missing=[]

            for skill in career["skills"]:

                if skill.lower() not in user_skills:

                    missing.append(skill)

            return "Missing Skills : " + ", ".join(missing)

    return "Role not found."

In [11]:
load_dotenv()
import requests, os
from bs4 import BeautifulSoup
from tavily import TavilyClient

tavily_client=TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(search_query: str) -> str:
    """
    Search the web for the latest career-related information.

    Use this tool when the user asks about:
    - Latest job openings
    - Career trends
    - Salary information
    - Certifications and online courses
    - Skills required for a specific career
    - Industry news
    - AI/ML, Data Science, Web Development, Cyber Security, etc.
    - Any information that is not available in the local dataset.

    Always use this tool for live or up-to-date information.
    """
    try:
        response = tavily_client.search(query=search_query, max_results=5)

        results = []

        for r in response.get("results", []):
            title = r.get("title", "No Title")
            url = r.get("url", "")
            snippet = r.get("content", "")[:200]

            results.append(
                f"Title: {title}\n"
                f"URL: {url}\n"
                f"Snippet: {snippet}"
            )

        return "\n\n---\n\n".join(results) if results else "No web results found."

    except Exception as e:
        return f"Web search error: {str(e)}"

In [12]:
@tool
def scrape_web_page(url: str) -> str:
    """
    Scrape a web page and extract useful career-related information.

    Use this tool when the user provides a URL containing:
    - Job listings
    - Career guidance articles
    - Online course pages
    - Certification details
    - Company career pages
    - Technology blogs
    - Salary reports
    - Skill requirement pages

    Always pass the complete URL, including https://

    Extract only the main readable text content and ignore
    advertisements, navigation bars, headers, footers, scripts,
    and other unnecessary page elements.
    """
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (AI Career Guidance Agent)"
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted HTML tags
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        text = soup.get_text(separator="\n", strip=True)

        return text[:3000]

    except Exception as e:
        return f"Error scraping {url}: {str(e)}"

print("Web tools loaded: web_search(), scrape_web_page()")

Web tools loaded: web_search(), scrape_web_page()


In [13]:
llm=ChatGroq(model="qwen/qwen3-32b")

In [ ]:
agent = create_agent(
    model=llm,
    tools=[
        search_career,
        salary_info,
        roadmap_generator,
        recommend_courses,
        skill_gap,
        web_search,
        scrape_web_page
    ],
    system_prompt="""
You are an AI Career Guidance Agent.

Your job is to help users choose the right career path.

Responsibilities:
- Recommend suitable career options.
- Suggest learning roadmaps.
- Recommend online courses and certifications.
- Analyze users' current skills and identify skill gaps.
- Provide salary insights for different career roles.
- Use web_search for the latest career trends, jobs, and certifications.
- Use scrape_web_page when the user provides a URL to a job posting, course page, or career article.
- Always use the appropriate tools before answering.
- Never make up information if a tool can provide it.
"""
)

In [25]:
q=input("enter your query")

In [26]:
def ask_q(q: str) -> str:
    config = {"configurable": {"thread_id": "session_1"}}

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": q
                }
            ]
        },
        config
    )

    return response["messages"][-1].content

In [27]:
print(ask_q(q))

Here's a comprehensive guide to becoming an AI Engineer based on the latest trends and requirements:

### 🚀 **AI Engineer Career Overview**
- **Skills Required**: Python, Machine Learning, Deep Learning, LLMs (Large Language Models)
- **Salary Range**: ₹8–20 LPA (India) | $138K+ median in the US
- **Key Trends (2024–2025)**:
  - High demand for AI/ML engineers in tech, healthcare, finance, and automotive sectors
  - Growing focus on GenAI (Generative AI), AI ethics, and MLOps
  - Remote and global hiring opportunities increasing

---

### 🛠️ **Learning Roadmap**
1. **Foundations**:
   - Learn Python (NumPy, Pandas, Scikit-learn)
   - Study Mathematics (Linear Algebra, Calculus, Probability)
2. **Core AI/ML**:
   - Take [Andrew Ng's Machine Learning course](https://www.coursera.org/learn/machine-learning)
   - Learn Deep Learning (TensorFlow/PyTorch) via [Deep Learning Specialization](https://www.coursera.org/specializations/deep-learning)
3. **Advanced Topics**:
   - Generative AI with